# Speed Limits and Road Safety: Assessing the Effects of Amsterdam's 30 km/h Policy

TODO:

- [ ] Figures
- [ ] 2.3.4
- [ ] 3.4
- [ ] Discussion
- [ ] Conclusion
- [ ] References
- [ ] Spellcheck

## 1. Introduction

### 1.1 Background & Motivation

Traffic safety is an important social issue, especially in busy cities such as Amsterdam, where many pedestrians, cyclists and vehicles come together. The speed of vehicles plays an important role, because a lower speed can reduce the chance and severity of the consequences. 

To improve traffic safety, on 8 December 2023, Amsterdam’s local authorities decreased the maximum speed from 50 km/h to 30 km/h on a large number of roads in the city. This measure was mostly meant to improve the safety of vulnerable traffic users, e.g. cyclists and pedestrians. 

Although it is expected that a lower maximum speed improves traffic safety, it is important to research whether this is actually visible in practice. Therefore, the development of the number and severity of traffic accidents in Amsterdam before and after the introduction of the measure is analyzed. Additionally, these developments are compared to nationwide trends to determine if these developments are specific for Amsterdam. 

### 1.2 Research Question & Objectives

The goal of this study is to investigate whether the introduction of the maximum speed limit of 30 km/h in Amsterdam is connected to changes in traffic safety. Not only the number of registered traffic accidents is investigated, but also their severity. The main research question is as follows: 

> “To what extent has the introduction of the 30 km/h speed limit in Amsterdam been associated with changes in traffic safety?”

To be able to answer this question, the study is divided into 4 subquestions:
1. Is there a greater effect on the roads with the changed speed limit from 50-30 km/h than on the roads without change?
2. Has the severity of the traffic accidents in Amsterdam changed since the change in speed limit?
3. How does a possible change compare to the nationwide trend?
4. What is the difference between the predicted trend of traffic accidents in Amsterdam if there was no intervention from 2023 until now compared to the real data? 

In this study, registered accidents from 2016-2025 are analysed. The period before the policy change is compared to the period after. In the dataset, what month the accident happened is missing. The speed limit was introduced in December 2023, so for that year, it is not possible to determine whether the accident happened before or after the new speed limit. Therefore, 2023 is viewed as a transition year. 

## 2. Data and Methodology

Three types of data are used: accident records, road segments with corresponding speed limits, and Dutch population records. Accidents are linked to road segments by segment ID (wegvak_ID).

### 2.1 Datasets

The following datasets are used:

Bestand geRegistreerde Ongevallen Nederland (BRON): Rijkwaterstaat’s record of all reported traffic accidents, from various sources. These go back to 2003, but the 2016–2025 dataset is used in this research. This dataset includes 1,237,603 accidents, with 66 columns for properties such as location, weather conditions, types of vehicles involved, severity, etc.
Nationaal Wegenbestand (NWB): a file containing every road segment in the Netherlands. The current and historical versions are used.
Amsterdam’s “30 km/u in de stad” map (amsterdam_dertig.json): a file containing each planned road type change. For each affected road segment, this lists the old speed limit, the new speed limit, segment ID, coordinates, and more.
Dutch population records: a file from Statistics Netherlands (CBS) which contains the average population numbers for the municipalities of Amsterdam, Rotterdam, The Hague, Utrecht, Eindhoven and Groningen from 2016 to 2025. The file also contains the average population of all municipalities in the Netherlands combined from 2016 to 2025.

### 2.2 Data Cleaning and Preparation

The BRON dataset contains more than a million accidents from across the whole nation, with tens of columns for each accident’s details. We start by filtering by accidents in Amsterdam, leaving 63,291 accidents. Next we remove the columns that are not relevant to this research, leaving 16 (out of 66); details such as wegvak_id, severity, recorded speed limit, etc. (see the file for more).

The main data preparation step is linking accidents to road groups: “50->30”, “stayed 50”, “always 30”, “mixed”, “other” (mostly highways), and “outside network”. However, a problem was identified: some accidents refer to old road segment IDs that no longer exist. Whenever changes are made to roads, the segments get new IDs, but the roads in Amsterdam’s 30 km/h map still have the IDs from when the map was created.

To match these accidents to Amsterdam’s roads, older versions of the NWB were used to find the coordinates of the road segment, which are then cross referenced against Amsterdam’s map (within 5 meters) to determine which road group the accident occurred on. As a second pass, they are also matched by street name to confirm. Junctions get the group of the main road that runs through them.

We were able to classify 99% of accidents in Amsterdam using these methods. The remaining 1% could not be matched because the locations in both datasets were too far apart, the segment ID was not found in any NWB version, or the main road type could not be inferred from a junction.

### 2.3 Data Analysis Methods

#### 2.3.1 Research Question 1

The roads that changed from 50 to 30 km/h were compared with the roads where the speed limit stayed the same. This was done with the difference-in-difference (DID) approach with the following steps:
1. Road groups: with the road map (amsterdam_dertig.json) the roads were split into the roads that went from 50 to 30 km/h and the control groups, the roads which stayed 50 and 30 km/h. Every accident was linked to one of the groups in wegvakken.ipynb.
2. Periods: Before is 2016-2022 and After is 2024-2025. The transition year 2023 was left out.
3. Pre-trends: it was checked whether the groups developed in the same way before 2023, because the DiD depends on it.
4. Difference-in-Difference: a quasi-Poisson regression on the yearly number of accidents per road group, with a dummy for every year and for every road group. The result is the incidence rate ratio (IRR): if it is below 1 it means fewer accidents on the changed road then expected. The model was also run with an extra trend for the changed road, as a robustness check.
5. Severity: a logistic regression on the share of serious accidents (injury or fatal).
6. Sensitivity analysis: the DiD was also done without the COVID years, with only the road segments, leaving the accidents on the junctions out because they can not be linked to one road group, and with only 2022 as Before.

#### 2.3.2 Research Question 2

To be able to answer the research question with the datasets, the severity of traffic accidents before and after the speed limit were compared, graphs were made and statistical tests were performed. The following steps were carried out:
1. Accident severity classification: the variable AP3_CODE was used to distribute the accidents in three categories: DOD (fatal), LET (injury) and UMS (only material damage).
2. Before-and-after comparison: The period of 2016-2022 was classified as Before and the period of 2024-2025 as After. The transition year 2023 was not included in this comparison. 
3. Severity distribution: The numbers and percentages per severity category were calculated to compare the distributions between the different periods.
4. Statistical testing: A chi-square test of independence was performed to investigate whether the difference was statistically significant. Cramér’s V was also calculated to determine how strong the connection was. 
5. Sensitivity analysis: Lastly, it was also determined if the results changed when the COVID-years were excluded.  

#### 2.3.3 Research Question 3
To assess how these changes compare to the nationwide rates of traffic accidents, the trendlines have been compared to two distinct groups. The first group consists of all the municipalities within the Netherlands, and the second group consists of the five largest cities in the Netherlands excluding Amsterdam. The trendlines of traffic accidents in Amsterdam and both of these groups have been visualised through the following steps:
1. Creation of additional dataframes: from the filtered datasets, dataframes were made for each distinct municipality and the Netherlands as a whole. This includes Amsterdam, Rotterdam, The Hague, Utrecht, Eindhoven, Groningen and the Netherlands. Additionally, the dataset containing Dutch population numbers was made into a useful, numerical dataframe. 
2. Accident severity classification: similarly as the first step of research question 2, the variable AP3_CODE was used to distribute the accidents in three categories for every dataframe: DOD (fatal), LET (injury) and UMS (only material damage).
3. Traffic accidents per capita: to be able to make a comparison the amount of traffic accidents needs to be adjusted for size, this is done with help of the Dutch population dataframe which contains the population numbers of the Netherlands and each municipality for the years 2016 to 2025.
4. Visualisation of results: the results have been displayed clearly in two types of line graph; one with the absolute amount of traffic accidents per capita and one with the amount of 

#### 2.3.4 Research Question 4
To estimate what would have happened without the 30 km/h limit, a trend was fitted on the years before the change and extended to 2024 and 2025. This predicted trend was then compared with the actual number of registered accidents. The following steps were carried out:
1. Accidents per year: the accidents were counted per year, because the dataset only gives the year of an accident. This was done for three series: all of Amsterdam, the city roads (every road group except “other”, which is mostly highways that were not part of the change) and the roads that changed from 50 to 30 km/h.
2. Pre-period: the trend was fitted on 2016-2019 and 2022. The COVID years 2020-2021 were left out because traffic was reduced in those years, and 2023 was left out as the transition year.
3. Trend model: a quasi-Poisson regression of the yearly number of accidents on the year, with a log link, so that the trend is a constant percentage change per year. A quasi-Poisson model was used because the differences between years are larger than a regular Poisson model assumes, so the uncertainty was scaled with an estimated dispersion factor. A separate model was fitted for each of the three series.
4. Prediction: the fitted trend was extended to 2024 and 2025, with a 95% prediction interval for each year. This interval includes both the uncertainty of the trend and the normal variation between years.
5. Comparison: the actual number of accidents was compared with the predicted number, as a percentage difference. It was also checked whether the actual number fell within the 95% prediction interval. If it did, the difference is no larger than normal variation between years.
6. Sensitivity analysis: because the trend is based on only five years, it depends on which years are used. The analysis was therefore repeated with three other pre-periods: 2016-2019 and 2022-2023 (with the transition year), 2016-2019 (only before COVID) and 2016-2022 (all years before the change, with the COVID years).


## 3. Results

### 3.1 Effect on the Changed Roads

The roads that changed from 50 to 30 km/h had fewer accidents after the new speed limit was in effect, but this drop is not larger than on the other roads.

| Road group | Accidents per year before | After | Change |
| ---------- | ------------------------- | ----- | ------ |
| 50 to 30   |                     1,864 | 1,541 |   -17% |
| Stayed 50  |                       694 |   698 |    +1% |
| Already 30 |                     1,623 | 1,366 |   -16% |

On the changed road the number of accidents per year went down 17%, while it stayed the same on the roads that stayed 50. However, the streets that were already 30 km/h show almost the same drop (-16%), even though nothing changed there.

[TODO: accidents per road group in Amsterdam]

Before 2023, the changed roads and the streets that were already 30 both went down faster than the roads that stayed 50. The pre-trend check confirms this: compared with the roads that stayed 50, the changed roads lost about 6% more accidents per year before the change (p < 0.001). Compared with the streets that were already 30, the changed roads went down about 4% per year slower (p = 0.003). So the trends were not parallel for either control group.

[TODO: difference-in-differences]

| Compared with | DiD: IRR (95% CI) | p | DiD with pre-trend: IRR (95% CI) | p |
| ------------- | ----------------- | - | -------------------------------- | - |
| Stayed 50 | 0.82 (0.67-1.01) | 0.06 | 1.15 (1.04-1.27) | 0.005 |
| Already 30 | 0.98 (0.82-1.17) | 0.84 | 0.79 (0.64-0.97) | 0.02 |

The difference-in-differences shows that, compared with the roads that stayed 50, the changed roads had 18% fewer accidents than expected (IRR 0.82, 95% CI 0.67-1.01, p = 0.06). This is just not significant. Compared with the streets that were already 30, there was almost no difference (IRR 0.98, p = 0.84).

When the different pre-trends are taken into account, the results point in opposite directions: 15% more accidents than expected compared with the roads that stayed 50 (IRR 1.15, p = 0.005), and 21% fewer compared with the streets that were already 30 (IRR 0.79, p = 0.02). The result therefore depends on which roads are used for the comparison.

In general, the changed roads had fewer accidents after the introduction of the 30 km/h limit, but they were already on a downward trend before. It can therefore not be concluded that the effect on these roads was greater than on the roads without a change.

To check the robustness of the result, the DiD was repeated 3 times with each one change in the data against the roads that stayed 50 km/h.

| Analysis | IRR (95% CI) | p |
| -------- | ------------ | - |
| Main analysis | 0.82 (0.67-1.01) | 0.06 |
| Without COVID years (2020-2021) | 0.80 (0.64-1.00) | 0.045 |
| Only road segments (no junctions) | 0.77 (0.62-0.97) | 0.02 |
| Pre-period 2022 only | 0.97 (0.92-1.02) | 0.21 |

Without the COVID years and with only road segments, the drop in changed roads is a bit larger (20-23%) and becomes significant. With only 2022 as the Before period, the effect almost disappears. This is probably caused by the different pre-trends, because the accidents on the changed roads were already going down faster than the roads that stayed 50.

### 3.2 Changes in Traffic Accident Severity

To investigate whether the severity of traffic accidents in Amsterdam changed after the introduction of the speed limit of 30 km/h, the distribution of the registered accidents before (2016-2022) and after (2024-2025) the policy change were compared. 

The first figure shows the relative changes of injury and fatal accidents in Amsterdam between 2016 and 2025. The year 2016 is used as a reference year, with an index value of 100. Because an indexed graph is used, the relative changes in both categories, before and after the new speed limit, can be compared to one another, despite the difference in absolute values. To accomplish this, different scales are used on the y-axis. 

[TODO: traffic accident severity in Amsterdam]

The number of accidents with injury decreased in the first years of the research period and reached the lowest level in 2020 and 2021. Since 2022, an increasing trend is again visible. In 2024 and 2025 the number of injury accidents was higher than in 2023, but the number remained beneath the level of 2016. The number of fatal accidents display larger fluctuations. Especially in 2024 and 2025, the index value was significantly higher than in the reference year 2016. However, it needs to be noted that fatal accidents are relatively rare, which results in small changes leading to considerable percentual differences.

[TODO: traffic accident severity distribution in Amsterdam]

The results show a change in the distribution of the severity of traffic accidents. Before the introduction of the measure, 82.9% of the accidents involved only material damage, against 80.25% after the measure. At the same time, the number of accidents with injury increased from 16.88% to 19.42%. The portion of fatal accidents increased from 0.22% to 0.34%. The second figure illustrates these differences in the distribution of the severity of the traffic accidents for both time periods. 

Because fatal accidents account for only a small percentage of all registered accidents, this category is separately displayed in the third figure. It shows that the amount of fatal accidents increased from 0.22% before the introduction of the speed limit to 0.34% after. Even though this is a vast increase relatively, the portion of fatal accidents in the total amount of registered accidents remains very small. 

[TODO: proportion of fatal traffic accidents in Amsterdam]

A chi-square test for independence showed that there was a statistically significant difference in the distribution of the severity before and after the introduction of the speed limit ($χ^2(2) = 48,223; p < 0.001$). However, the value of Cramér’s V amounted only to 0,0292, which indicates a very weak coherence between the period and the severity of the accidents. 

To check whether the COVID-19 pandemic affected these findings, an additional analysis was performed where the years 2020 and 2021 were left out of consideration. The results remained statistically significant ($p < 0.001$), with a similar effect size (Cramér’s $V = 0.0270$). 

In general, the portion of registered accidents with injuries or fatalities was higher after the introduction of the 30 km/h speed limit. This observed cohesion was very weak, though. Besides, based on these results it cannot be confirmed that the change in speed limit was the cause for the observed difference.

### 3.3 Comparison to the Nationwide Trend

[TODO:  graphs]

### 3.4 Comparison to Predicted Trend

## 4. Discussion

### 4.1 Interpretation of Results

The main research question was whether the 30 km/h limit, introduced in December 2023, affected traffic accidents in Amsterdam. The results show some signs of fewer accidents, but they are too small and too uncertain to conclude the limit caused them.

For subquestion 1 the roads that changed from 50 to 30 km/h had 17% fewer accidents per year after the change while the roads that stayed 50 km/h stayed the same. Compared with these roads the drop was thus 18% larger (p = 0.06). However, the streets which were already 30 km/h showed almost the same drop of 16 percent and before 2023 the number of accidents already developed differently on the changed roads from the other road groups. This may be because traffic moved to the 50 km/h roads, because drivers slowed down across the city, or because this decrease had already started before 2023. Without more data like traffic counts or speed measurements we can not test the first two explanations.

For subquestion 2, the share of accidents with an injury or a fatality rose from 17% to 20% (Cramér’s V = 0,029). This happened because injury accidents fell by 11%, while accidents with only material damage fell by 7%.

For subquestion 3, the number of accidents went down by about 4% in Amsterdam, while it went up by 4% for the Netherlands as a whole. Injury accidents also rose less in Amsterdam (11%) than nationwide (37%). The rise found in subquestion 2 is thus not specific to Amsterdam. 

For subquestion 4, the actual number of accidents in 2024 and 2025 was 4 to 6 percent below the predicted trend citywide, and 3 to 9 percent below it on changed roads with all values falling within the 95% prediction interval. This means that differences of this size can also happen by chance in a normal year. As in subquestion 1 the changed roads show a slightly larger decrease than the rest of the city, but the difference is too small to be called an effect of the speed limit.

Together, the subquestions show slightly fewer accidents than expected after the change.Still, the differences are small, mostly not significant and sometimes follow a trend seen before 2023. With only two years of data after the change, an effect of the 30 km/h limit can not be confirmed.

### 4.2 Limitations

The results have some limitations. First, the BRON data only gives the year of an accident. Because the limit changed on 8 December 2023, the whole year 2023 had to be left out. This gives us 11 months less information on the before and less than a month of information on the after. This matters because only two years are available after the change, and 2023 is the year closest to the new situation.

Secondly, there are no traffic counts for Amsterdam city streets. If traffic moved from the 30 km/h streets to the roads that stayed 50 km/h, fewer accidents could simply mean less traffic, and our data can not show this. In addition, accidents with only material damage are not always registered by the police, so a change in registration can look like a change in safety. 

Thirdly, COVID-19 reduced traffic in 2020 and 2021. We checked the results without these years, but travel behaviour may have changed after the pandemic. This also makes the predicted trend in subquestion 4 less certain. 

Lastly, not every accident could be linked to a road, and accidents could be matched to a road segment, and accidents at junctions were assigned to a road group based on the meeting roads. Some accidents may therefore be in the wrong group. Other changes in the same period, such as road redesigns or the growing number of e-bikes, may explain part of the results as well.

### 4.3 Recommendations

Repeating the analysis when more years of data are available would give more certain results. Adding traffic counts and measured speeds for the different road groups would show whether traffic moved to other roads and whether drivers actually drive slower.

## 5. Conclusion

## 6. Contribution & AI Statement

As a group, we collectively decided the course of our project. We decided on the main and sub research questions together. Also, searching for datasets and deciding which periods we would investigate is also something we did with the whole group. After that, we decided to divide the subquestions, so every group member had the possibility to write code and learn something from it. To finish the project, we wrote the report together. 

- Jaden Accord: finding datasets, data cleaning and preparation, writing code, research question 4, writing “Data and Methodology” section, checking the report.
- Nienke van Slooten: writing code, making graphs, creating planning for the project, research question 2, writing and checking report. 
- Jelle Kock: writing code, making code, research question 1, writing the discussion.
- Liam van de Luijtgaarden: writing code, research question 3, writing the conclusion, organising repository, checking the report

In this report, generative AI was used to help to write code when errors were encountered, when we did not know how we could accomplish something in VS code or to clean up our written code. It was also used to structure the report language and translation wise.

## References

TODO: format these properly

- https://maps.amsterdam.nl/30km/
- https://maps.amsterdam.nl/30km_kaart/
- https://data.overheid.nl/dataset/9841-verkeersongevallen---bestand-geregistreerde-ongevallen-nederland
- https://downloads.rijkswaterstaatdata.nl/bron/01-01-2016_31-12-2025.zip
- https://data.overheid.nl/dataset/14767-nwb---wegen#panel-resources
- https://downloads.rijkswaterstaatdata.nl/nwb-wegen/geogegevens/shapefile/Nederland_total/01-01-2016.zip (01-01-2016 tot 01-01-2025)
- https://opendata.cbs.nl/#/CBS/nl/dataset/70072ned/table 